# Tarea 1: Predicción de resultados del fútbol uruguayo

In [1]:
# %pip install pandas numpy matplotlib scikit-learn

In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer

--- 
### 1. Carga del Dataset y Descripción de Atributos

In [3]:
DATASET_FILE = "./futbol_uruguayo.csv" 

dataset = pd.read_csv(DATASET_FILE)
dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national


#### Descripción de los atributos:

| Atributo | Descripción |
| :--- | :--- |
| **`home`** | Nombre del equipo local (no necesariamente único) |
| **`away`** | Nombre del equipo visitante (no necesariamente único) |
| **`date`** | Fecha del partido |
| **`gh`** | Goles del equipo local (incluyendo tiempo extra y penales) |
| **`ga`** | Goles del equipo visitante (incluyendo tiempo extra y penales) |
| **`full_time`** | "F"=el partido terminó en 90', "E"=tiempo extra, "P"=penales |
| **`competition`** | Nombre del país de la liga o nombre de la competición int. |
| **`home_ident`** | Identificador único del equipo local |
| **`away_ident`** | Identificador único del equipo visitante |
| **`home_country`** | País del equipo local |
| **`away_country`** | País del equipo visitante |
| **`home_code`** | Código de país del equipo local |
| **`away_code`** | Código de país del equipo visitante |
| **`home_continent`** | Continente del equipo local |
| **`away_continent`** | Continente del equipo visitante |
| **`continent`** | Continente de la competición |
| **`level`** | "national"= liga local, "international"= copa internacional |

--- 
### 2. Definición de la Variable Objetivo (`ganador`)

El objetivo del modelo es predecir el resultado final de un partido de fútbol, clasificándolo en una de tres categorías posibles: victoria local, victoria visitante o empate.

Dado que el dataset original no incluye directamente una columna de resultado, se deduce la variable objetivo **`ganador`** mediante la comparación de los goles anotados por el equipo local (`gh`) y el visitante (`ga`):

* Si $\text{gh} > \text{ga} \implies$ **`L`**
* Si $\text{ga} > \text{gh} \implies$ **`V`**
* Si $\text{gh} == \text{ga} \implies$ **`E`**

Concretado lo anterior, la información que proveen los atributos "gh" y "ga" ya se ve contemplada por la variable objetivo. Por lo tanto, su presencia en el dataset no agrega significancia al entrenamiento del modelo y se deben remover ambos atributos del dataset.

In [4]:
dataset["ganador"] = np.select(
    [
        dataset["gh"] > dataset["ga"],
        dataset["gh"] < dataset["ga"],
        dataset["gh"] == dataset["ga"],
    ],
    [
        "L",
        "V",
        "E",
    ],
    default="sin_dato",
)

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level,ganador
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,V
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,E
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L


--- 
### 3. Selección de Atributos

En esta etapa del preprocesamiento se realiza una selección de los atributos del dataset con el objetivo de maximizar la capacidad de aprendizaje del modelo en la clasificación de las instancias. Para ello, se examina cada atributo de forma individual para evaluar si contribuye de manera significativa al entrenamiento o si, dadas sus características, puede ser removido sin perjudicar el desempeño.

* **Atributos constantes:** No contribuyen al aprendizaje del modelo debido a que mantienen el mismo valor para todas las instancias del dataset (varianza cero).
* **Atributos redundantes:** Representan valores equivalentes dentro del dataset, por lo que basta con conservar uno de ellos.

#### A. Detección de Atributos Constantes

In [5]:
# Detección de Atributos Constantes
constantes = dataset.nunique(dropna=False)[dataset.nunique(dropna=False) == 1]
print("Atributos constantes detectados:")
print(constantes)

Atributos constantes detectados:
competition       1
home_country      1
away_country      1
home_code         1
away_code         1
home_continent    1
away_continent    1
continent         1
level             1
dtype: int64


Los atributos constantes son los siguientes:
* `competition`, `home_country`, `away_country` (Todos refieren a Uruguay).
* `home_code`, `away_code` (Código constante `UY`).
* `home_continent`, `away_continent`, `continent` (Todos refieren a `South America`).
* `level` (Constante con el valor `national`).

Estos atributos no serán incluidos en los conjuntos de entrenamiento y evaluación.

#### B. Detección de Atributos Redundantes

In [6]:
# Verificar cuántos identificadores tiene cada nombre de equipo, y viceversa
home_name_to_id = dataset.groupby("home")["home_ident"].nunique(dropna=False)
home_id_to_name = dataset.groupby("home_ident")["home"].nunique(dropna=False)

away_name_to_id = dataset.groupby("away")["away_ident"].nunique(dropna=False)
away_id_to_name = dataset.groupby("away_ident")["away"].nunique(dropna=False)

print(f"Cada valor de home se corresponde a un solo valor de home_ident, y viceversa: ", (home_name_to_id == 1).all()  & (home_id_to_name == 1).all())
print(f"Cada valor de away se corresponde a un solo valor de away_ident, y viceversa: ", (away_name_to_id == 1).all()  & (away_id_to_name == 1).all())

Cada valor de home se corresponde a un solo valor de home_ident, y viceversa:  True
Cada valor de away se corresponde a un solo valor de away_ident, y viceversa:  True


Existe una correspondencia 1:1, mantener ambas variables introduciría información redundante. Elegimos quedarnos con `home` y `away` y se descartan sus identificadores (`home_ident` y `away_ident`).

#### C. Descomposición del atributo `date`

El atributo `date` se divide en los atributos `day`, `month` y `year` para representar sus componentes por separado. Esta transformación facilita que el modelo identifique patrones relacionados con el momento del año en que se disputó el partido, como diferencias entre meses, temporadas o períodos históricos. Además, evita tratar cada fecha completa como un valor independiente, lo que podría dificultar el aprendizaje. Una vez extraídos estos componentes, se debe remover el atributo `date` original para evitar mantener información redundante.

En el flujo final, este procedimiento será ejecutado dentro del `Pipeline`.

In [7]:
dataset["date"] = pd.to_datetime(dataset["date"])
dataset["day"] = dataset["date"].dt.day
dataset["month"] = dataset["date"].dt.month
dataset["year"] = dataset["date"].dt.year

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,...,home_code,away_code,home_continent,away_continent,continent,level,ganador,day,month,year
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,V,5,3,1932
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,E,5,3,1932
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932


---
### 4. Análisis de Balance y Estratificación:

In [8]:
print(f"Cantidad total de instancias: {dataset.shape[0]}")
print(f"Cantidad total de atributos: {dataset.shape[1]}")

Cantidad total de instancias: 15207
Cantidad total de atributos: 21


In [9]:
# Cálculo de la dispersión de instancias respecto a las clases
dataset["ganador"].value_counts(normalize=True).mul(100).round(2)

ganador
L    44.35
V    28.20
E    27.44
Name: proportion, dtype: float64

La distribución de la clase objetivo muestra las siguientes proporciones:
* **`L`**: **44.35%**
* **`V`**: **28.20%**
* **`E`**: **27.44%**

Aunque existe un predominio de las victorias locales, la distribución no presenta un desbalance crítico (como ocurriría en escenarios con clases minoritarias $< 5\%$), por lo que no se requiere la aplicación de técnicas de mitigación como *SMOTE* o *undersampling*. 

Sin embargo, para evitar que una división puramente aleatoria, se aplica un muestreo estratificado (`stratify=dataset_Y`). De esta manera, se garantiza que tanto el conjunto de entrenamiento como el de evaluación mantengan proporciónes similares a las originales del dataset para cada clase.

---
### 5. División del Conjunto de Datos (`train_test_split`)

Se debe dividir el conjunto dataset de la siguiente forma:

Conjunto de entrenamiento los partidos jugados hasta el año 2023 inclusive y como conjunto de evaluación los partidos jugados en 2024 y 2025.


In [10]:
# Instancias con partidos jugados hasta el año 2023 inclusive
df_entrenamiento = dataset[dataset['year'] <= 2023]
X_train = df_entrenamiento.drop(columns=["ganador", "gh", "ga"])
Y_train = df_entrenamiento["ganador"]

df_evaluacion = dataset[dataset['year'] > 2023]
X_test = df_evaluacion.drop(columns=["ganador", "gh", "ga"])
Y_test = df_evaluacion["ganador"]

print(f"Dimensiones de X_train (Entrenamiento): {X_train.shape}")
print(f"Dimensiones de X_test (Evaluación):    {X_test.shape}")

Dimensiones de X_train (Entrenamiento): (14734, 18)
Dimensiones de X_test (Evaluación):    (473, 18)


In [11]:
numeric_features = ['year', 'month', 'day']
categorical_features = ['home', 'away', 'full_time']

# Pipeline para los atributos numéricos
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer()) # Reemplaza los valores NaN por la media (mean) por defecto
])

# Pipeline para los atributos categóricos
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(fill_value='unknown',strategy='constant')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessing = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    remainder='drop' # Elimina automáticamente todas las demás columnas no especificadas
)


In [12]:
MIN_INFO_GAIN = 0.005

In [13]:

pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', DecisionTreeClassifier(
        criterion='entropy',
        min_impurity_decrease=MIN_INFO_GAIN,
        random_state=62
    ))
])

---
### X. Clasificador base

Se debe implementar un codificador base que no aprende de los atributos del partido, sino que para cada equipo, calcula la proporción de partidos ganados de los últimos 10 de dicho equipo, y toma como ganador al que tenga mayor proporción.

In [17]:
from sklearn.base import BaseEstimator, ClassifierMixin

class BaseClasificator(BaseEstimator, ClassifierMixin):

    def __init__(self, years=10, col_local="home", col_away="away", col_year="year"):
        self.years = years
        self.col_local = col_local
        self.col_away = col_away
        self.col_year = col_year

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = pd.Series(np.asarray(y), index=X.index)
        self.classes_ = np.unique(y)

        max_year = int(X[self.col_year].max())
        self.last_year = max_year - self.years + 1
        in_window = X[self.col_year] >= self.last_year
        X_v = X[in_window]
        y_v = y[in_window]

        played = pd.concat([X_v[self.col_local], X_v[self.col_away]]).value_counts()

        winned = (
            X_v.loc[y_v == "L", self.col_local].value_counts()
            .add(X_v.loc[y_v == "V", self.col_away].value_counts(), fill_value=0)
            .reindex(played.index, fill_value=0)
        )

        self.proportion = (winned / played).rename("wins_prop")
        self.unknown_proportion = float(winned.sum() / played.sum())

        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        prop_local = X[self.col_local].map(self.proportion).fillna(self.unknown_proportion)
        prop_away = X[self.col_away].map(self.proportion).fillna(self.unknown_proportion)

        return np.where(prop_local.to_numpy(float) >= prop_away.to_numpy(float), "L", "V")

In [18]:
from sklearn.metrics import classification_report, confusion_matrix

baseline = BaseClasificator()
baseline.fit(X_train, Y_train)

print(f"Ventana utilizada: {baseline.last_year}–{int(X_train['year'].max())}")
print(baseline.proportion.sort_values(ascending=False).head(10).round(3))

Y_pred_base = baseline.predict(X_test)
print(classification_report(Y_test, Y_pred_base, zero_division=0))
print(confusion_matrix(Y_test, Y_pred_base, labels=["E", "L", "V"]))

Ventana utilizada: 2014–2023
Nacional                0.608
CA Penarol              0.547
Liverpool               0.431
Defensor Sporting       0.415
Montevideo Wanderers    0.392
River Plate             0.374
Cerro Largo FC          0.358
Danubio                 0.348
Torque FC               0.341
CA Progreso             0.341
Name: wins_prop, dtype: float64
              precision    recall  f1-score   support

           E       0.00      0.00      0.00       132
           L       0.51      0.65      0.57       190
           V       0.42      0.65      0.51       151

    accuracy                           0.47       473
   macro avg       0.31      0.43      0.36       473
weighted avg       0.34      0.47      0.39       473

[[  0  64  68]
 [  0 123  67]
 [  0  53  98]]
